In [ ]:
from model_arch import *

@dataclass
class NERConfig(LMConfig):
    num_classes: int = 25

class ThirdDecoderForTokenClassification(nn.Module):
    def __init__(self, cfg: NERConfig, pretrained_lm: ThirdDecoderLM):
        super().__init__()
        self.num_classes = cfg.num_classes
        self.transformer = pretrained_lm
        self.transformer.head = nn.Identity()
        self.classifier = nn.Linear(cfg.d_model, cfg.num_classes)

    def forward(self, idx):
        hidden_states = self.transformer(idx)
        logits = self.classifier(hidden_states)
        
        return logits

In [ ]:
import torch.nn.functional as F

def kd_loss(student_logits, teacher_logits, labels, temperature=2.0, alpha=0.5):
    mask = labels != -100
    s_logits = student_logits[mask]
    t_logits = teacher_logits[mask]
    y = labels[mask]

    ce_loss = F.cross_entropy(s_logits, y)
    student_soft = F.log_softmax(s_logits / temperature, dim=-1)
    teacher_soft = F.softmax(t_logits / temperature, dim=-1)
    kl_loss = F.kl_div(student_soft, teacher_soft, reduction="batchmean") * (temperature ** 2)

    return (alpha * ce_loss) + ((1.0 - alpha) * kl_loss)

In [ ]:
import pickle
from torch.utils.data import Dataset, DataLoader

with open("label_mappings.pkl", "rb") as f:
    a = pickle.load(f)
    id2label = a['id2label']
    label2id = a['label2id']

with open("integer_chunked_ner_folds.pkl", "rb") as f:
    all_folds = pickle.load(f)

train_chunks = []
for i in range(4):
    train_chunks.extend(all_folds[i])

val_chunks = all_folds[4]

class FastNERDataset(Dataset):
    def __init__(self, data):
        self.data = data
        
    def __len__(self):
        return len(self.data)
        
    def __getitem__(self, idx):
        item = self.data[idx]
        return {
            "input_ids": torch.tensor(item["input_ids"], dtype=torch.long),
            "labels": torch.tensor(item["labels"], dtype=torch.long)
        }

train_loader = DataLoader(FastNERDataset(train_chunks), batch_size=8, shuffle=True)
val_loader = DataLoader(FastNERDataset(val_chunks), batch_size=8, shuffle=False)

In [ ]:
from torch import optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
base_cfg = LMConfig(vocab_size=50260)
ner_cfg = NERConfig(vocab_size=50260, num_classes=25)

teacher_base = ThirdDecoderLM(base_cfg)
teacher_model = ThirdDecoderForTokenClassification(ner_cfg, teacher_base)
teacher_model.load_state_dict(torch.load("best_ner_model_teacher.pt", map_location=device))
teacher_model.to(device)
teacher_model.eval()
for param in teacher_model.parameters():
    param.requires_grad = False

student_base = ThirdDecoderLM(base_cfg)
student_base.load_state_dict(torch.load("swa.pt", map_location=device))

keep_layers = [0, 2, 4, 6]
student_base.cfg.n_layers = len(keep_layers)
student_base.blocks = nn.ModuleList([student_base.blocks[i] for i in keep_layers])

ner_model = ThirdDecoderForTokenClassification(ner_cfg, student_base)
ner_model.classifier.load_state_dict(teacher_model.classifier.state_dict())
ner_model.to(device)

optimizer = optim.AdamW(ner_model.parameters(), lr=5e-5)

In [ ]:
for epoch in range(10):
    ner_model.train()
    total_train_loss = 0
    
    for batch_idx, batch in enumerate(train_loader):
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)
        
        student_logits = ner_model(input_ids) 
        
        with torch.no_grad():
            teacher_logits = teacher_model(input_ids)
            
        loss = kd_loss(
            student_logits.view(-1, ner_model.num_classes),
            teacher_logits.view(-1, teacher_model.num_classes),
            labels.view(-1)
        )
        
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item()
            
    avg_train_loss = total_train_loss / len(train_loader)
    
    torch.save(ner_model.state_dict(), "kd_checkpoint.pt")
    print(f"Epoch {epoch} | Train Loss: {avg_train_loss:.4f}")